# Churn Prediction for a Telecommunications Provider

You are a data analyst in a telecommunications company. Your company is facing a high churn rate and you are tasked with creating a model to predict which customers are most likely to churn next. 

In [217]:
# Define your imports here
import numpy as np
import pandas as pd

# We will use the balanced accuracy score to evaluate our models
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression


## Load data

In [218]:
customers = pd.read_csv("customers.csv")
payment_info = pd.read_csv("payment_info.csv")
service_options = pd.read_csv("service_options.csv")
churn = pd.read_csv("churn_analysis.csv")

### Helper Functions

You can use the following helper functions in your submission. 

In [219]:
def one_hot_encoding(df: pd.DataFrame) -> pd.DataFrame:
    """ A function to automatically apply one-hot encoding to all string and categorical variables in a dataframe df, excluding the customer_id column. 
    
    Example usage: df_encoded = one_hot_encoding(df)
    
    """
    
    cat_cols = df.select_dtypes(include=["object", "category"]).columns
    cat_cols = cat_cols.drop("customer_id", errors="ignore")

    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

    # convert True/False → 1/0
    bool_cols = df.select_dtypes(include="bool").columns
    df[bool_cols] = df[bool_cols].astype(int)

    return df


## Description of the available data:

*customers.csv*: Customer profile and service subscription information
- customer_id: Unique identifier for each customer
- gender: Customer’s gender
- age: Customer’s age (in years)
- partner: Indicates whether the customer has a partner (Yes/No)
- number_of_dependents: Number of dependents associated with the customer
- married: Indicates whether the customer is married (Yes/No)
- online_security: Subscription to an online security add-on (Yes/No)
- online_backup: Subscription to an online backup add-on (Yes/No)
- device_protection: Subscription to a device protection add-on (Yes/No)
- premium_tech_support: Subscription to premium technical support (Yes/No)
- streaming_tv: Subscription to streaming TV services (Yes/No)
- streaming_movies: Subscription to streaming movie services (Yes/No)
- streaming_music: Subscription to streaming music services (Yes/No)
- internet_type: Type of internet service (e.g., Fiber Optic, DSL, Cable)

*payment_info.csv*: Customer payment, billing, and revenue information
- customer_id: Unique identifier for each customer
- contract: Type of customer contract
- paperless_billing: Indicates whether the customer uses paperless billing (Yes/No)
- payment_method: Method used for payment
- monthly_charges: Recurring monthly charges
- avg_monthly_long_distance_charges: Average monthly long-distance charges
- total_charges: Total charges incurred by the customer
- total_refunds: Total amount refunded to the customer
- total_extra_data_charges: Total charges for extra data usage
- total_long_distance_charges: Total long-distance charges
- total_revenue: Total revenue generated by the customer
- unit: Unit in which the revenues are expressed (e.g., cents, dollars)


*service_options.csv*: Customer service usage, tenure, and marketing information
- customer_id: Unique identifier for each customer
- tenure: Customer tenure (number of months with the company)
- internet_service: Indicates whether the customer has internet service (Yes/No)
- phone_service: Indicates whether the customer has phone service (Yes/No)
- multiple_lines: Indicates whether the customer has multiple service lines (Yes/No)
- avg_monthly_gb_download: Average monthly data download in gigabytes (GB)
- unlimited_data: Indicates whether the customer has an unlimited data plan (Yes/No)
- offer: Most recent marketing offer accepted by the customer (None, Offer A–E)
- referred_a_friend: Indicates whether the customer referred a friend (Yes/No)
- number_of_referrals: Number of referrals made by the customer

*churn.csv*: Customer churn outcome information
- customer_id: Unique identifier for each customer
- churn: Indicates whether the customer churned in the following quarter (Yes/No)



__Task__: Some customer_ids do not have a churn label. Use the given data sets to develop a predictive model to predict the churn of exactly these customers. Remember, churn is binary.  

# Data Preparation

## Data Cleaning

In [220]:

customers['gender'] = customers['gender'].map({'Female': 0, 'F': 0, 'f':0,'Male': 1, 'M': 1, 'm':1})
customers['partner'] = customers['partner'].map({'Yes': 1,'No': 0})
customers['married'] = customers['married'].map({'Yes': 1,'No': 0})
customers['online_security'] = customers['online_security'].map({'Yes': 1,'No': 0})
customers['online_backup'] = customers['online_backup'].map({'Yes': 1,'No': 0})
customers['device_protection'] = customers['device_protection'].map({'Yes': 1,'No': 0})
customers['premium_tech_support'] = customers['premium_tech_support'].map({'Yes': 1,'No': 0})
customers['streaming_tv'] = customers['streaming_tv'].map({'Yes': 1,'No': 0})
customers['streaming_movies'] = customers['streaming_movies'].map({'Yes': 1,'No': 0})
customers['streaming_music'] = customers['streaming_music'].map({'Yes': 1,'No': 0})
customers=one_hot_encoding(customers)
customers = customers.dropna() 
payment_info=payment_info.dropna()
payment_info=one_hot_encoding(payment_info)
(payment_info[payment_info['unit_dollars']==0]['total_revenue'])/=100.0
service_options.dropna()
service_options= one_hot_encoding(service_options)


service_options.head()

/tmp/ipykernel_1220912/1616870477.py:15: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  (payment_info[payment_info['unit_dollars']==0]['total_revenue'])/=100.0
/tmp/ipykernel_1220912/1616870477.py:15: SettingWithCopyWarning: 
A value is tryin

,customer_id,tenure,avg_monthly_gb_download,number_of_referrals,internet_service_Yes,phone_service_Yes,multiple_lines_Yes,unlimited_data_Yes,offer_Offer B,offer_Offer C,offer_Offer D,offer_Offer E,referred_a_friend_Yes
0,0002-ORFBO,9,16,2,1,1,0,1,0,0,0,0,1
1,0003-MKNFE,9,10,0,1,1,1,0,0,0,0,0,0
2,0004-TLHLJ,4,30,0,1,1,0,1,0,0,0,1,0
3,0011-IGKFF,13,4,1,1,1,0,1,0,0,1,0,1
4,0013-EXCHZ,3,11,3,1,1,0,1,0,0,0,0,1


## Merge datasets

In [ ]:

merged_data = pd.merge(
    payment_info, 
    service_options,
    left_on="customer_id", 
    right_on="customer_id",
    how="inner"
)
merged_data2 = pd.merge(
    merged_data, 
    customers,
    left_on="customer_id", 
    right_on="customer_id",
    how="inner"
)
merged_data2=merged_data2.dropna()
merged_data3 = pd.merge(
    merged_data2, 
    churn,
    left_on="customer_id", 
    right_on="customer_id",
    how="inner"
)




data=merged_data3
data.head()

,customer_id,monthly_ charges,avg_monthly_long_distance_charges,total_charges,total_refunds,total_extra_data_charges,total_long_distance_charges,total_revenue,contract_One Year,contract_Two Year,...,streaming_tv,streaming_movies,streaming_music,internet_type_DSL,internet_type_DSL,internet_type_Fiber,internet_type_Fiber Optic,internet_type_Optic,internet_type_cable,churn
0,0002-ORFBO,65.6,42.39,593.30,0.00,0.0,381.51,974.81,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,NaN
1,0003-MKNFE,5990.0,10.69,542.40,38.33,10.0,96.21,610.28,0.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0004-TLHLJ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0011-IGKFF,98.0,27.82,1237.85,0.00,0.0,361.66,1599.51,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,0013-EXCHZ,83.9,7.38,267.40,0.00,0.0,22.14,289.54,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0


# Modelling 

## Train / Test Split (optional)

In [222]:
data_train = data[data['churn'].notna()].copy()
data_train=data_train.dropna()
data_pred = data[data['churn'].isna()].copy()
feature_cols = [c for c in data.columns if c not in ['customer_id', 'churn']]
X = data_train[feature_cols]

y = data_train['churn'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)



## Model Fitting

In [223]:
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)
y_val_pred = model.predict(X_test)
score = balanced_accuracy_score(y_test, y_val_pred)
print(score)
X_sub = data_pred[feature_cols]
final_predictions = model.predict(X_sub)
submission = pd.DataFrame({'customer_id': data_pred['customer_id'], 'churn': final_predictions})
submission.to_csv("submission_rf.csv", index=False)
    

0.6711640100963616


/home/jason-mann/Desktop/Buisness_analytics/baml-venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## Prediction

Create the final submission by generating a csv file with exactly the following format:

```csv
id,prediction
0002-ORFBO,0
0004-TLHLJ,0
0014-BMAQU,0
0023-UYUPN,0
0023-XUOPT,0
0027-KWYKW,0
0031-PVLZI,0
0042-RLHYP,0
...
```

The id column contains the customer_id, and the prediction column contains the predicted label.

You can find a sample submission in which all predicted labels are 0 in the file "sample_prediction.csv".

In [ ]:
sample_prediction = pd.read_csv("sample_prediction.csv")
sample_prediction.head()

,id,prediction
0,0002-ORFBO,0
1,0004-TLHLJ,0
2,0014-BMAQU,0
3,0023-UYUPN,0
4,0023-XUOPT,0
